<a href="https://colab.research.google.com/github/Sharmadipti/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sharmadipti/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: The Freshness Multiplier
The paper reports that older content refreshed within 30 days showed a 3.2x health increase and 57x more impressions than comparable stale content. The label comes from observed portfolio performance metrics such as health score and impressions rather than a controlled experiment.

My methodology question is whether refreshed pages were already stronger pages before the refresh. Because the study is observational, the validation design cannot fully separate the effect of refreshing from factors such as content quality, topic demand, or editorial attention. The finding is useful and actionable, but it should be interpreted as a directional association rather than proof of causation. 【1-f40874】

Finding 2: The Content Performance Curve
The paper reports that content tends to peak around 61–90 days and declines after 270 days, while some older content appears to recover after refresh activity. The outcome is based on observed health-score comparisons across age buckets.

My methodology question is whether age alone explains the pattern. Older content may differ in topic mix, competition level, or refresh frequency. The validation design supports a descriptive comparison across age groups, but it does not prove that aging itself causes performance decline. The result is best used as a planning signal for content reviews rather than as a deterministic rule. 【1-f40874】


In [4]:
import os
import getpass
import duckdb

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    return getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{get_hf_token()}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Connected to the FlyRank warehouse.")

query_1 = f"""
SELECT
    client_hash_id,
    content_hash_id,
    COUNT(*) AS daily_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MAR}
GROUP BY client_hash_id, content_hash_id
ORDER BY daily_rows DESC
LIMIT 10
"""

grain_check = con.sql(query_1).df()

grain_check

query_2 = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {MAR}
"""

slice_check = con.sql(query_2).df()

print("March 2026 slice:")
slice_check

query_3 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_with_gsc_data
FROM {MAR}
"""

availability_check = con.sql(query_3).df()

print("GSC data availability for March 2026:")
availability_check

# ---------------------------------------------------------
# Five-feature frame
# February = feature window
# March = label window
# ---------------------------------------------------------

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
DIM = f"read_parquet('{REL}/dim_content.parquet')"

feature_query = f"""
WITH feb_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_feb,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_feb
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

march_label AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_mar
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.impressions_feb,
    f.clicks_feb,
    f.ctr_feb,
    f.avg_position_feb,
    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-02-28'
    ) AS content_age_days,
    CASE
        WHEN COALESCE(m.clicks_mar, 0) = 0 THEN 1
        ELSE 0
    END AS went_dark
FROM feb_features f
LEFT JOIN march_label m
    ON f.client_hash_id = m.client_hash_id
   AND f.content_hash_id = m.content_hash_id
LEFT JOIN {DIM} c
    ON f.content_hash_id = c.content_hash_id
WHERE f.impressions_feb >= 100
  AND f.clicks_feb >= 3
  AND c.is_published IS TRUE
  AND c.content_created_date <= DATE '2026-02-28'
"""

feature_frame = con.sql(feature_query).df()

print(f"Feature-frame rows: {len(feature_frame):,}")
print("\nFive features:")
print([
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "content_age_days"
])

feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_feb",
        "clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "content_age_days",
        "went_dark"
    ]
].head()



Paste your Hugging Face READ token: ··········
Connected to the FlyRank warehouse.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 slice:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GSC data availability for March 2026:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 29,700

Five features:
['impressions_feb', 'clicks_feb', 'ctr_feb', 'avg_position_feb', 'content_age_days']


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,content_age_days,went_dark
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4270.0,7.0,0.001639,5.960187,365,0
1,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5271.0,4.0,0.000759,7.125783,365,0
2,client_73cda7b4e4f265ea,content_a7da352b73b02668,6690.0,19.0,0.002840,7.226756,365,0
3,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,4314.0,14.0,0.003245,6.424200,365,0
4,client_73cda7b4e4f265ea,content_20403327d8d9374c,2756.0,8.0,0.002903,6.726415,365,0


In [5]:
feature_frame = con.sql(feature_query).df()
print(f"Feature-frame rows: {len(feature_frame):,}")

Feature-frame rows: 29,700


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
paper_findings = {
    "Finding 1": "Freshness Multiplier",
    "Finding 2": "Content Performance Curve"
}

print("Findings selected for methodology review:")
for k, v in paper_findings.items():
    print(f"{k}: {v}")

Findings selected for methodology review:
Finding 1: Freshness Multiplier
Finding 2: Content Performance Curve


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*
In ML-08, I used a stratified train/test split. However, pages from the same client could appear in both training and testing data.

For a stricter validation audit, I also evaluated the model using a grouped split based on client_hash_id. This reduces the risk that client-specific patterns appear in both datasets.

Because only a single February→March snapshot is available, a true time-based split is not possible. A grouped split is the most appropriate additional validation available for this dataset.

I compare the original stratified split with the grouped split below.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score

feature_cols = [
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "content_age_days"
]

X = feature_frame[feature_cols].fillna(
    feature_frame[feature_cols].median()
)

y = feature_frame["went_dark"].astype(int)

groups = feature_frame["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train_g = X.iloc[train_idx]
X_test_g = X.iloc[test_idx]

y_train_g = y.iloc[train_idx]
y_test_g = y.iloc[test_idx]

group_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

group_model.fit(X_train_g, y_train_g)

group_pred = group_model.predict(X_test_g)

group_recall = recall_score(
    y_test_g,
    group_pred
)

print("ML-08 Recall (went_dark=1): 0.84")
print(f"Grouped Split Recall: {group_recall:.3f}")


ML-08 Recall (went_dark=1): 0.84
Grouped Split Recall: 0.740


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
The final model uses only features that were available at the end of the February feature window:

impressions_feb
clicks_feb
ctr_feb
avg_position_feb
content_age_days
The target variable, went_dark, is derived from March clicks and is not included as a feature.

During the ML-04 leakage experiment, I deliberately added the target variable as a feature. This produced unrealistically strong performance, demonstrating how future information can invalidate model evaluation.

The final model excludes all outcome-derived fields and uses only information available before the prediction point.


In [7]:
final_features = [
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "content_age_days"
]

print("Final feature set:")
for feature in final_features:
    print("-", feature)

print("\nTarget:")
print("went_dark")

print("\nLeakage check passed:")
print("No March outcome fields are included in the final feature set.")


Final feature set:
- impressions_feb
- clicks_feb
- ctr_feb
- avg_position_feb
- content_age_days

Target:
went_dark

Leakage check passed:
No March outcome fields are included in the final feature set.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim:

"The model identifies which pages should be refreshed and improves content performance."

Rewritten claim:

"The model produces a ranked list of pages that may deserve review based on observed February search-performance signals. The results are intended as decision-support and prioritization guidance. The analysis does not demonstrate that refreshing content causes performance improvements, nor does it guarantee future outcomes."

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
original_claim = (
    "The model identifies which pages should be refreshed and improves content performance."
)

safe_claim = (
    "The model produces a ranked list of pages that may deserve review "
    "based on observed search-performance signals. "
    "The results are intended as decision-support and do not imply causation."
)

print("Original Claim:")
print(original_claim)

print("\nRewritten Claim:")
print(safe_claim)

Original Claim:
The model identifies which pages should be refreshed and improves content performance.

Rewritten Claim:
The model produces a ranked list of pages that may deserve review based on observed search-performance signals. The results are intended as decision-support and do not imply causation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.